In [1]:
import pandas as pd
import numpy as np
import os
import glob
from sklearn.model_selection import train_test_split
#from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
#from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight # <-- Thêm thư viện này
from sklearn.pipeline import Pipeline
import numpy as np
import joblib # Thư viện chuyên dùng để lưu model ML
import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub

In [ ]:

OUTPUT_DIR = './data/artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True) 

# Download latest version
path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")

print("Path to dataset files:", path)

files = os.listdir(path)

csv_files = [f for f in files if f.endswith('.csv')]


# Danh sách 17 features bạn yêu cầu
selected_features = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean',
    'Packet Length Std', 'SYN Flag Count', 'ACK Flag Count',
    'FIN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'URG Flag Count'
]


Path to dataset files: /home/na/.cache/kagglehub/datasets/chethuhn/network-intrusion-dataset/versions/1


In [ ]:

# 1. Đọc file CSV (Ví dụ lấy file đầu tiên)

df_list = []
print(f"--- Đang đọc và lọc {len(csv_files)} files ---")
    
for file in csv_files:
    temp_df = pd.read_csv(path + '/' + file)
    temp_df.columns = temp_df.columns.str.strip()
        
    cols_to_keep = selected_features + ['Label']
    cols_present = [col for col in cols_to_keep if col in temp_df.columns]
        
    df_list.append(temp_df[cols_present])

df = pd.concat(df_list, ignore_index=True)
#df = pd.read_csv(file_path)

df.columns = df.columns.str.strip()

target_column = 'Label' 
all_needed_columns = selected_features + [target_column]

# Lọc lấy các cột cần thiết
df= df[all_needed_columns].copy()

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print("Kích thước dữ liệu sau khi lọc:", df.shape)
df.head()


In [ ]:
print("--- KÍCH THƯỚC DỮ LIỆU ---")
print(f"Số dòng: {df.shape[0]}, Số cột: {df.shape[1]}")

print("\n--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---")
display(df.head()) # Dùng display() trong Kaggle/Jupyter sẽ hiển thị bảng đẹp hơn print()

print("\n--- THỐNG KÊ KIỂU DỮ LIỆU & GIÁ TRỊ THIẾU ---")
# Lấy danh sách các cột có chứa giá trị NaN (nếu có)
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])
# 3. Chuẩn hóa tên (Phòng trường hợp cột tên là 'label' viết thường hoặc 'Class')
if 'label' in df.columns:
    df.rename(columns={'label': 'Label'}, inplace=True)
elif 'Class' in df.columns:
    df.rename(columns={'Class': 'Label'}, inplace=True)
    
print("\n--- PHÂN BỔ CÁC LỚP NHÃN (LABEL) ---")
if 'Label' in df.columns:
    print(df['Label'].value_counts())
else:
    print("Không tìm thấy cột 'Label', vui lòng kiểm tra lại tên cột.")


In [ ]:
# 2. Làm sạch nhãn (Dùng Regex để sửa lỗi ký tự lạ)
print("--- Đang dọn dẹp lỗi Encoding của nhãn ---")
if 'Label' in df.columns:
    df['Label'] = df['Label'].replace(to_replace=r'.*Brute Force.*', value='Web Attack - Brute Force', regex=True)
    df['Label'] = df['Label'].replace(to_replace=r'.*XSS.*', value='Web Attack - XSS', regex=True)
    df['Label'] = df['Label'].replace(to_replace=r'.*Sql Injection.*', value='Web Attack - Sql Injection', regex=True)
        
    df.dropna(subset=['Label'], inplace=True)
    

In [ ]:
def optimize_logistic_for_skewed_data(df, selected_features):
    print("\n" + "="*50)
    print("BẮT ĐẦU TỐI ƯU HÓA LOGISTIC REGRESSION (BẢN TINH CHỈNH)")
    print("="*50)

    # 1. Dọn rác
    print("--- 1. Đang dọn dẹp giá trị NaN, Inf và Duplicates ---")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()
    df = df.drop_duplicates()

    # 2. Gom nhóm & Lọc nhãn
    print("--- 2. Đang gom nhóm và làm sạch nhãn ---")
    df['Label'] = df['Label'].replace(to_replace=r'^Web Attack.*', value='Web Attack', regex=True)
    
    label_counts = df['Label'].value_counts()
    valid_labels = label_counts[label_counts > 500].index
    df = df[df['Label'].isin(valid_labels)]

    # 3. Undersampling BENIGN
    print("--- 3. Đang Undersampling lớp BENIGN ---")
    benign_indices = df[df['Label'] == 'BENIGN'].index
    np.random.seed(42)
    # Cắt giảm 70% BENIGN để cân đối lại không gian dữ liệu
    drop_indices = np.random.choice(benign_indices, size=int(len(benign_indices) * 0.7), replace=False)
    df = df.drop(drop_indices)

    # 4. CẮT TỈA OUTLIERS
    print("--- 4. Đang khống chế giá trị ngoại lệ (Clipping Outliers) ---")
    for col in selected_features:
        upper_limit = df[col].quantile(0.99)
        df[col] = np.where(df[col] > upper_limit, upper_limit, df[col])

    # 5. Mã hóa & Tách biến
    print("--- 5. Đang chuẩn bị tập Train/Test ---")
    le = LabelEncoder()
    y = le.fit_transform(df['Label'])
    X = df[selected_features]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # ----------------------------------------------------------------------
    # TÍNH NĂNG MỚI: TỰ ĐỘNG LÀM MỀM TRỌNG SỐ (SMOOTHED CLASS WEIGHTS)
    # ----------------------------------------------------------------------
    print("--- 6. Đang tính toán Trọng số thông minh (Smoothed Weights)... ---")
    classes = np.unique(y_train)
    # Tính trọng số gắt (mặc định của Sklearn)
    raw_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    
    # Kỹ thuật làm mềm: Lấy căn bậc 2 để thu hẹp khoảng cách giữa nhãn lớn và nhãn nhỏ.
    # Nó giúp Precision tăng mạnh nhưng không làm mất đi Recall của các lớp hiếm.
    smoothed_weights = np.sqrt(raw_weights) 
    
    # Ép kiểu thành Dictionary để truyền vào LogisticRegression
    custom_weight_dict = dict(zip(classes, smoothed_weights))
    # ----------------------------------------------------------------------

    # 6. Pipeline Cốt lõi
    print("--- 7. Đang thiết lập Pipeline... ---")
    pipeline = Pipeline([
        ('scaler', StandardScaler()), 
        ('classifier', LogisticRegression(
            solver='lbfgs',
            class_weight=custom_weight_dict,  # <-- Dùng trọng số vừa tính toán thay vì 'balanced'
            C=0.5,                            # Tăng phạt L2 (Regularization) để giảm nhiễu (chống học vẹt lớp nhỏ)
            max_iter=3000,           
            n_jobs=-1,               
            random_state=42
        ))
    ])

    print("--- 8. Đang huấn luyện mô hình... ---")
    pipeline.fit(X_train, y_train) 
    
    print("--- 9. Đang dự đoán trên tập Test ---")
    y_pred = pipeline.predict(X_test)
    
    print("\n" + "="*60)
    print("BÁO CÁO KẾT QUẢ LOGISTIC REGRESSION (ĐÃ TỐI ƯU)")
    print("="*60)
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

    export_dir = './data/models'

    os.makedirs(export_dir, exist_ok=True) 

    model_filename = os.path.join(export_dir, 'logistic_ids_model.pkl')
    encoder_filename = os.path.join(export_dir, 'label_encoder.pkl')

    joblib.dump(pipeline, model_filename)
    joblib.dump(le, encoder_filename)

    print(f"Thành công! Các file đã được đóng gói gọn gàng tại: {export_dir}")
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix - SVC (RBF Kernel)')
    plt.ylabel('Thực tế (True Label)')
    plt.xlabel('Dự đoán (Predicted Label)')
    plt.tight_layout()

    plt.savefig( OUTPUT_DIR +  "/logistic_regression.png")
    plt.show()

# Gọi hàm
optimize_logistic_for_skewed_data(df, selected_features)